In [ ]:

from fastapi import (
    APIRouter,
    HTTPException,
    status,
    Depends,
)

from app.api.auth import get_current_user


router = APIRouter(
    prefix="/mastery",
    tags=["Mastery"],
)


def get_database():
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


def verify_project_ownership(
    database,
    project_id: str,
    user_id: str,
):
    project = database.collection("projects").find_one(
        {
            "id": project_id,
            "user_id": user_id,
        }
    )

    if project is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Project not found.",
        )

    return project


def clean(document):
    result = dict(document)
    result.pop("_id", None)
    return result


@router.get("/project/{project_id}")
async def get_project_mastery(
    project_id: str,
    current_user=Depends(get_current_user),
):
    """Return concept-level mastery for an authorized project."""

    database = get_database()

    verify_project_ownership(
        database,
        project_id,
        current_user.id,
    )

    mastery_documents = database.collection("mastery").find(
        {
            "project_id": project_id,
            "user_id": current_user.id,
        }
    ).sort(
        "score",
        1,
    )

    items = [
        clean(document)
        for document in mastery_documents
    ]

    if items:
        average_score = sum(
            float(item.get("score", 0.0))
            for item in items
        ) / len(items)
    else:
        average_score = 0.0

    return {
        "project_id": project_id,
        "user_id": current_user.id,
        "concept_count": len(items),
        "average_score": round(average_score, 4),
        "items": items,
    }


@router.get("/project/{project_id}/concept/{concept_id}")
async def get_concept_mastery(
    project_id: str,
    concept_id: str,
    current_user=Depends(get_current_user),
):
    """Return mastery for one authorized concept."""

    database = get_database()

    verify_project_ownership(
        database,
        project_id,
        current_user.id,
    )

    mastery = database.collection("mastery").find_one(
        {
            "project_id": project_id,
            "user_id": current_user.id,
            "concept_id": concept_id,
        }
    )

    if mastery is None:
        return {
            "project_id": project_id,
            "user_id": current_user.id,
            "concept_id": concept_id,
            "score": 0.0,
            "confidence": 0.0,
            "trend": "stable",
            "assessment_count": 0,
            "recent_evidence": [],
        }

    return clean(mastery)
